# 01. Harmonic Oscillator via QFT Split-Operator Evolution

This notebook uses the standard periodic QFT/FFT split-operator method for a harmonic oscillator on a finite box. The exact reference is reconstructed from the analytical harmonic-oscillator eigenbasis. The finite periodic box is treated as a numerical approximation and checked through boundary and spatial-convergence diagnostics.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from qftsplit.core import (
    build_periodic_resource_circuit,
    configure_matplotlib,
    dirichlet_midpoint_grid,
    draw_and_save_circuit,
    fft_momentum_grid,
    fidelity_summary,
    harmonic_potential,
    periodic_grid,
    plot_convergence,
    plot_density_snapshots,
    plot_fidelity_vs_gate_count,
    plot_fidelity,
    qft_split_circuit_validation,
    run_harmonic_case,
    run_infinite_well_case,
    save_dataframe,
    save_publication_figure,
    sine_mode_energies,
    sine_transform_validation,
    transpile_and_extract_metrics,
)

FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

In [2]:
# Main parameters
N = 64
x_min = -8.0
x_max = 8.0
hbar = 1.0
m = 1.0
omega = 1.0

x0 = 2.0
k0 = 0.0
sigma = 1.0

t_max = 2.0 * np.pi
r = 200
snapshot_count = 5

reference_grid_size = 4096
reference_tail_tolerance = 1e-10
reference_basis_cap_min = 96

fidelity_sweep_r = [50, 100, 150, 200, 250, 300, 400]
spatial_convergence_N = [32, 64, 128]

In [3]:
result = run_harmonic_case(
    grid_size=N,
    x_left=x_min,
    x_right=x_max,
    x0=x0,
    k0=k0,
    sigma=sigma,
    t_max=t_max,
    steps=r,
    mass=m,
    omega=omega,
    hbar=hbar,
    reference_grid_size=reference_grid_size,
    reference_tail_tolerance=reference_tail_tolerance,
    reference_basis_cap=max(reference_basis_cap_min, 2 * N),
)
summary = fidelity_summary(result)

display(Markdown("## Main Run Summary"))
display(pd.DataFrame([summary]))
display(
    Markdown(
        f"Reference basis kept **{result['reference_data']['n_keep']}** states out of cap "
        f"**{result['reference_data']['basis_cap']}**, with discarded tail estimate "
        f"**{result['reference_data']['tail_weight']:.2e}**."
    )
)

## Main Run Summary

,minimum_fidelity,final_time_fidelity,mean_fidelity,max_split_norm_error,max_reference_norm_error,max_edge_probability
0,1.0,1.0,1.0,1.398881e-14,4.440892e-16,5.084142e-07


Reference basis kept **64** states out of cap **128**, with discarded tail estimate **9.37e-11**.

In [4]:
parameter_df = pd.DataFrame(
    [
        {
            "system": "harmonic_oscillator",
            "numerical_transform": "periodic_QFT_or_FFT",
            "boundary_model": "finite_periodic_box_checked_by_edge_probability",
            "N": N,
            "n_qubits": int(np.log2(N)),
            "domain": f"[{x_min}, {x_max})",
            "x0": x0,
            "k0": k0,
            "sigma": sigma,
            "t_max": t_max,
            "r": r,
            "dt": result["dt"],
            "hbar": hbar,
            "m": m,
            "omega": omega,
            "reference_grid_size": result["reference_data"]["dense_grid_size"],
            "reference_basis_cap": result["reference_data"]["basis_cap"],
            "reference_basis_kept": result["reference_data"]["n_keep"],
            "reference_tail_weight": result["reference_data"]["tail_weight"],
            "max_edge_probability": result["max_edge_probability"],
            "fidelity_sweep_r": ",".join(str(value) for value in fidelity_sweep_r),
            "spatial_convergence_N": ",".join(str(value) for value in spatial_convergence_N),
        }
    ]
)
fidelity_df = pd.DataFrame(
    {
        "time": result["times"],
        "fidelity": result["fidelity"],
        "split_norm": result["split_norms"],
        "reference_norm": result["reference_norms"],
    }
)

save_dataframe(parameter_df, TABLES_DIR, "harmonic_parameters.csv")
save_dataframe(fidelity_df, TABLES_DIR, "harmonic_fidelity_vs_time.csv")

display(Markdown("## Simulation Parameters"))
display(parameter_df)

## Simulation Parameters

,system,numerical_transform,boundary_model,N,n_qubits,domain,x0,k0,sigma,t_max,...,hbar,m,omega,reference_grid_size,reference_basis_cap,reference_basis_kept,reference_tail_weight,max_edge_probability,fidelity_sweep_r,spatial_convergence_N
0,harmonic_oscillator,periodic_QFT_or_FFT,finite_periodic_box_checked_by_edge_probability,64,6,"[-8.0, 8.0)",2.0,0.0,1.0,6.283185,...,1.0,1.0,1.0,4096,128,64,9.371548e-11,5.084142e-07,"50,100,150,200,250,300,400","32,64,128"


In [5]:
snapshot_figure = plot_density_snapshots(
    x=result["x"],
    times=result["times"],
    reference_states=result["reference_states"],
    split_states=result["split_states"],
    snapshot_count=snapshot_count,
    title="Harmonic oscillator probability density",
)
fidelity_figure = plot_fidelity(result["times"], result["fidelity"], "Harmonic oscillator fidelity")

save_publication_figure(snapshot_figure, FIGURES_DIR, "harmonic_density_snapshots")
save_publication_figure(fidelity_figure, FIGURES_DIR, "harmonic_fidelity_vs_time")
plt.close(snapshot_figure)
plt.close(fidelity_figure)

print("Saved harmonic density and fidelity figures.")

Saved harmonic density and fidelity figures.


In [6]:
sweep_records = []
for r_value in fidelity_sweep_r:
    local = run_harmonic_case(
        grid_size=N,
        x_left=x_min,
        x_right=x_max,
        x0=x0,
        k0=k0,
        sigma=sigma,
        t_max=t_max,
        steps=int(r_value),
        mass=m,
        omega=omega,
        hbar=hbar,
        reference_grid_size=reference_grid_size,
        reference_tail_tolerance=reference_tail_tolerance,
        reference_basis_cap=max(reference_basis_cap_min, 2 * N),
    )
    local_summary = fidelity_summary(local)
    sweep_records.append(
        {
            "system": "harmonic_oscillator",
            "r": int(r_value),
            "dt": local["dt"],
            "final_time_fidelity": local_summary["final_time_fidelity"],
            "minimum_fidelity": local_summary["minimum_fidelity"],
            "mean_fidelity": local_summary["mean_fidelity"],
        }
    )

fidelity_sweep_df = pd.DataFrame(sweep_records)
save_dataframe(fidelity_sweep_df, TABLES_DIR, "harmonic_fidelity_vs_r.csv")
display(Markdown("## Trotter-Step Convergence"))
display(fidelity_sweep_df)

## Trotter-Step Convergence

,system,r,dt,final_time_fidelity,minimum_fidelity,mean_fidelity
0,harmonic_oscillator,50,0.125664,0.999927,0.999905,0.999968
1,harmonic_oscillator,100,0.062832,0.999995,0.999994,0.999998
2,harmonic_oscillator,150,0.041888,0.999999,0.999999,1.000000
3,harmonic_oscillator,200,0.031416,1.000000,1.000000,1.000000
4,harmonic_oscillator,250,0.025133,1.000000,1.000000,1.000000
5,harmonic_oscillator,300,0.020944,1.000000,1.000000,1.000000
6,harmonic_oscillator,400,0.015708,1.000000,1.000000,1.000000


In [7]:
convergence_records = []
for n_value in spatial_convergence_N:
    local = run_harmonic_case(
        grid_size=int(n_value),
        x_left=x_min,
        x_right=x_max,
        x0=x0,
        k0=k0,
        sigma=sigma,
        t_max=t_max,
        steps=r,
        mass=m,
        omega=omega,
        hbar=hbar,
        reference_grid_size=reference_grid_size,
        reference_tail_tolerance=reference_tail_tolerance,
        reference_basis_cap=max(reference_basis_cap_min, 2 * int(n_value)),
    )
    local_summary = fidelity_summary(local)
    convergence_records.append(
        {
            "system": "harmonic_oscillator",
            "N": int(n_value),
            "n_qubits": int(np.log2(n_value)),
            "dx": local["dx"],
            "final_time_fidelity": local_summary["final_time_fidelity"],
            "minimum_fidelity": local_summary["minimum_fidelity"],
            "max_edge_probability": local_summary["max_edge_probability"],
        }
    )

spatial_convergence_df = pd.DataFrame(convergence_records)
save_dataframe(spatial_convergence_df, TABLES_DIR, "harmonic_spatial_convergence.csv")
convergence_figure = plot_convergence(
    spatial_convergence_df,
    x_column="N",
    y_column="final_time_fidelity",
    title="Harmonic oscillator spatial convergence",
    xlabel="Grid points, N",
)
save_publication_figure(convergence_figure, FIGURES_DIR, "harmonic_spatial_convergence")
plt.close(convergence_figure)

display(Markdown("## Spatial Convergence"))
display(spatial_convergence_df)

## Spatial Convergence

,system,N,n_qubits,dx,final_time_fidelity,minimum_fidelity,max_edge_probability
0,harmonic_oscillator,32,5,0.500,0.999945,0.999945,7.570419e-05
1,harmonic_oscillator,64,6,0.250,1.000000,1.000000,5.084142e-07
2,harmonic_oscillator,128,7,0.125,1.000000,1.000000,2.538832e-08
